# HazeWise activity exposure check

Enter an activity's minute ventilation (VE) and duration, choose a venue and time, then run the check. Blank environmental overrides are filled from Open-Meteo forecasts at the selected coordinates. Venue search uses OneMap when `ONEMAP_TOKEN` is set; otherwise enter coordinates directly.

The PM2.5 result is an estimate of inhaled mass, not the amount deposited in the lungs or a medical diagnosis. Cigarette equivalents are only a rough comparison based on an explicit configurable assumption. Risk levels below are simple planning rules, not official health thresholds.

In [10]:
import json
import math
import os
import re
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
import requests
import ipywidgets as widgets
from IPython.display import display, Markdown

OPEN_METEO_AQ = "https://air-quality-api.open-meteo.com/v1/air-quality"
OPEN_METEO_WX = "https://api.open-meteo.com/v1/forecast"
ONEMAP_SEARCH = "https://www.onemap.gov.sg/api/common/elastic/search"
DATA_GOV_POLL = "https://api-open.data.gov.sg/v1/public/api/datasets/{}/poll-download"
SPORTSG_DATASET = "d_9b87bab59d036a60fad2a91530e10773"
NPARKS_DATASET = "d_14d807e20158338fd578c2913953516e"
SINGAPORE_TZ = ZoneInfo("Asia/Singapore")

# Planning assumption for the optional comparison. Change this to suit your chosen source.
PM25_UG_PER_CIGARETTE = 12.0

# --- GeoJSON in-memory cache (keyed by dataset_id) ---
_geojson_cache: dict[str, dict] = {}
CACHE_DIR = Path(".")
CACHE_FILES = {
    SPORTSG_DATASET: CACHE_DIR / "cache_sportsg.json",
    NPARKS_DATASET: CACHE_DIR / "cache_nparks.json",
}

# --- Indoor facility classification pattern ---
INDOOR_KEYWORDS = re.compile(
    r"\b(gym|fitness|indoor|sports?\s*hall|squash|badminton|table\s*tennis|studio|swimming|pool|bowling|climbing|dojo|martial\s*arts)\b",
    re.IGNORECASE,
)

# --- Waking-hour constraints for alternative_time ---
WAKING_HOUR_START = 6    # 06:00 SGT
WAKING_HOUR_END   = 22   # 22:00 SGT

location_text = widgets.Text(value="East Coast Park, Singapore", description="Venue", layout=widgets.Layout(width="600px"))
latitude = widgets.FloatText(value=1.3010, description="Latitude")
longitude = widgets.FloatText(value=103.9120, description="Longitude")
activity = widgets.Dropdown(options=["Outdoor run", "Outdoor walk", "Cycling", "Other"], value="Outdoor run", description="Activity")
ve_l_min = widgets.FloatText(value=60.0, description="VE (L/min)")
duration_min = widgets.IntSlider(value=60, min=5, max=240, step=5, description="Minutes")
when_local = widgets.Text(value=(datetime.now(SINGAPORE_TZ).replace(minute=0, second=0, microsecond=0).strftime("%Y-%m-%d %H:%M")), description="Local time", placeholder="YYYY-MM-DD HH:MM")
pm25_override = widgets.FloatText(value=-1.0, description="PM2.5 override", step=1.0)
heat_override = widgets.FloatText(value=-1.0, description="Heat °C override", step=0.5)
uv_override = widgets.FloatText(value=-1.0, description="UV override", step=0.5)
rain_override = widgets.FloatText(value=-1.0, description="Rain % override", step=1.0)
run_button = widgets.Button(description="Assess exposure", button_style="primary", icon="check")
result_box = widgets.Output()

display(widgets.VBox([location_text, widgets.HBox([latitude, longitude]), activity, ve_l_min, duration_min, when_local,
                      widgets.HTML("<small>Set a measured value to override the forecast. Use -1 to keep the forecast for any field; zero is accepted as a real reading.</small>"),
                      widgets.HBox([pm25_override, heat_override, uv_override, rain_override]), run_button, result_box]))

In [11]:
def onemap_search(query):
    token = os.getenv("ONEMAP_TOKEN")
    if not token:
        return None
    response = requests.get(ONEMAP_SEARCH, params={"searchVal": query, "returnGeom": "Y",
                             "getAddrDetails": "Y", "pageNum": 1},
                            headers={"Authorization": token}, timeout=20)
    response.raise_for_status()
    body = response.json()
    if body.get("error") or not body.get("results"):
        return None
    return body["results"][0]

def resolve_location(query, fallback_lat, fallback_lon):
    result = onemap_search(query)
    if result:
        return {"name": result.get("SEARCHVAL", query), "address": result.get("ADDRESS", query),
                "lat": float(result["LATITUDE"]), "lon": float(result["LONGITUDE"]), "source": "OneMap"}
    return {"name": query, "address": query, "lat": float(fallback_lat), "lon": float(fallback_lon),
            "source": "entered coordinates (OneMap token unavailable or no match)"}

def forecast_at(lat, lon, when):
    aq_params = {"latitude": lat, "longitude": lon, "hourly": "pm2_5", "timezone": "Asia/Singapore", "forecast_days": 7}
    wx_vars = "apparent_temperature,temperature_2m,uv_index,precipitation_probability"
    wx_params = {"latitude": lat, "longitude": lon, "hourly": wx_vars, "timezone": "Asia/Singapore", "forecast_days": 7}
    aq_response = requests.get(OPEN_METEO_AQ, params=aq_params, timeout=25)
    wx_response = requests.get(OPEN_METEO_WX, params=wx_params, timeout=25)
    aq_response.raise_for_status(); wx_response.raise_for_status()
    aq, wx = aq_response.json(), wx_response.json()
    aq_times = pd.to_datetime(aq["hourly"]["time"])
    wx_times = pd.to_datetime(wx["hourly"]["time"])
    requested = pd.Timestamp(when.replace(tzinfo=None))
    if requested < aq_times.min() or requested > aq_times.max() or requested < wx_times.min() or requested > wx_times.max():
        raise ValueError("Requested time is outside the available forecast horizon.")
    aq_idx = int((aq_times - requested).to_series().abs().argmin())
    wx_idx = int((wx_times - requested).to_series().abs().argmin())
    return {"pm25": aq["hourly"]["pm2_5"][aq_idx],
            "heat": wx["hourly"]["apparent_temperature"][wx_idx],
            "temp": wx["hourly"]["temperature_2m"][wx_idx],
            "uv": wx["hourly"]["uv_index"][wx_idx],
            "rain": wx["hourly"]["precipitation_probability"][wx_idx],
            "aq_times": aq_times, "aq_values": aq["hourly"]["pm2_5"],
            "wx_times": wx_times, "wx_values": wx["hourly"]}

# ---------- GeoJSON caching ----------

def load_geojson_dataset(dataset_id):
    """Return parsed GeoJSON, using in-memory then on-disk cache before hitting the API."""
    # 1. In-memory hit
    if dataset_id in _geojson_cache:
        return _geojson_cache[dataset_id]

    # 2. On-disk hit
    cache_path = CACHE_FILES.get(dataset_id)
    if cache_path and cache_path.exists():
        with open(cache_path, "r", encoding="utf-8") as fh:
            data = json.load(fh)
        _geojson_cache[dataset_id] = data
        return data

    # 3. Network fetch
    response = requests.get(DATA_GOV_POLL.format(dataset_id), timeout=30)
    response.raise_for_status()
    body = response.json()
    if body.get("code") != 0 or not body.get("data", {}).get("url"):
        raise RuntimeError(body.get("errMsg", "Dataset download is not available"))
    data_response = requests.get(body["data"]["url"], timeout=60)
    data_response.raise_for_status()
    data = data_response.json()

    # Persist to disk and memory
    _geojson_cache[dataset_id] = data
    if cache_path:
        with open(cache_path, "w", encoding="utf-8") as fh:
            json.dump(data, fh)
    return data

def distance_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    dlat, dlon = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return 2 * radius * math.asin(math.sqrt(a))

# ---------- Indoor classification ----------

def classify_indoor(venue_name, details=""):
    """Return True if the venue name or details suggest an indoor / sheltered facility."""
    text = f"{venue_name} {details}"
    return bool(INDOOR_KEYWORDS.search(text))

def nearby_facilities(lat, lon, dataset_id, kind, radius_km=12, limit=8):
    geo = load_geojson_dataset(dataset_id)
    rows = []
    for feature in geo.get("features", []):
        coords = (feature.get("geometry") or {}).get("coordinates")
        props = feature.get("properties") or {}
        if not coords or len(coords) < 2:
            continue
        flon, flat = float(coords[0]), float(coords[1])
        km = distance_km(lat, lon, flat, flon)
        if km <= radius_km:
            if kind == "ActiveSG / SportSG":
                name = props.get("VENUE", "SportSG facility")
                detail = props.get("DETAILS", "")
            else:
                name = props.get("NAME") or props.get("CLASS") or "NParks facility"
                detail = props.get("CLASS", "")
            is_indoor = classify_indoor(name, detail)
            rows.append({"type": kind, "venue": name, "distance_km": km,
                         "lat": flat, "lon": flon, "details": detail,
                         "is_indoor": is_indoor})
    return sorted(rows, key=lambda row: row["distance_km"])[:limit]

def assess_risk(pm25, heat_c, uv, rain_pct):
    reasons = []
    if pm25 >= 35.5: reasons.append("PM2.5 is high (≥35.5 µg/m³)")
    elif pm25 >= 12.1: reasons.append("PM2.5 is elevated (≥12.1 µg/m³)")
    if heat_c >= 35: reasons.append("apparent temperature is very hot (≥35°C)")
    elif heat_c >= 32: reasons.append("apparent temperature is hot (≥32°C)")
    if uv >= 8: reasons.append("UV index is very high (≥8)")
    elif uv >= 6: reasons.append("UV index is high (≥6)")
    if rain_pct >= 70: reasons.append("rain probability is high (≥70%)")
    if pm25 >= 55.5 or heat_c >= 35 or (uv >= 8 and rain_pct >= 70):
        level = "High"
    elif pm25 >= 12.1 or heat_c >= 32 or uv >= 6 or rain_pct >= 50:
        level = "Moderate"
    else:
        level = "Lower"
    return level, reasons

# ---------- Alternative time with waking-hour and ±3 h preference ----------

def alternative_time(conditions, duration, preferred_hour=None):
    """Return the best forecast slots in the next 48 h, restricted to waking hours.

    If *preferred_hour* is given (0–23), slots within ±3 hours of that hour
    on any day are boosted (sorted first at equal severity).
    """
    times = conditions["wx_times"]
    choices = []
    start = conditions["when"]
    for i, timestamp in enumerate(times):
        ts = pd.Timestamp(timestamp).to_pydatetime().replace(tzinfo=SINGAPORE_TZ)
        if ts <= start or ts > start + pd.Timedelta(hours=48):
            continue
        # Waking-hour filter
        if not (WAKING_HOUR_START <= ts.hour < WAKING_HOUR_END):
            continue
        pm_idx = int((conditions["aq_times"] - pd.Timestamp(ts).tz_localize(None)).to_series().abs().argmin())
        pm = conditions["aq_values"][pm_idx]
        heat = conditions["wx_values"]["apparent_temperature"][i]
        uv = conditions["wx_values"]["uv_index"][i]
        rain = conditions["wx_values"]["precipitation_probability"][i]
        if None in (pm, heat, uv, rain): continue
        severity = max(pm / 12.0, heat / 32.0, uv / 6.0, rain / 70.0)
        # Prefer times within ±3 h of the user's intended workout hour
        if preferred_hour is not None:
            hour_diff = min(abs(ts.hour - preferred_hour), 24 - abs(ts.hour - preferred_hour))
            near_preferred = 0 if hour_diff <= 3 else 1
        else:
            near_preferred = 0
        choices.append((near_preferred, severity, ts, pm, heat, uv, rain))
    return sorted(choices, key=lambda row: (row[0], row[1]))[:3]

# ---------- Indoor-prioritised venue alternatives ----------

def alternative_venues(lat, lon, when, prefer_indoor=False):
    """Return nearby venue candidates, optionally prioritising indoor facilities."""
    candidates = nearby_facilities(lat, lon, SPORTSG_DATASET, "ActiveSG / SportSG", limit=8)
    candidates += nearby_facilities(lat, lon, NPARKS_DATASET, "NParks", limit=5)

    if prefer_indoor:
        # Indoor first, then outdoor; within each group sort by distance
        candidates.sort(key=lambda r: (not r["is_indoor"], r["distance_km"]))
    else:
        candidates.sort(key=lambda r: r["distance_km"])

    picks = candidates[:6]
    scored = []
    for item in picks:
        forecast = forecast_at(item["lat"], item["lon"], when)
        pm, heat, uv, rain = forecast["pm25"], forecast["heat"], forecast["uv"], forecast["rain"]
        item["risk"], item["drivers"] = assess_risk(pm, heat, uv, rain)
        item["conditions"] = {"pm25": pm, "heat": heat, "uv": uv, "rain": rain}
        scored.append(item)
    order = {"Lower": 0, "Moderate": 1, "High": 2}
    return sorted(scored, key=lambda row: (order[row["risk"]], not row["is_indoor"], row["distance_km"]))

# ---------- LLM coaching summary ----------

def generate_llm_summary(summary_payload):
    """Call an LLM to generate a friendly coaching summary.

    Uses the ``google.genai`` client when ``LLM_API_KEY`` is set.
    Returns ``None`` if the key is absent or the call fails.
    """
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        return None

    system_prompt = (
        "You are HazeWise Coach, a friendly Singapore sports-safety advisor. "
        "The user has already calculated deterministic metrics (inhaled dose, "
        "cigarette equivalents, risk level, alternative times, and venue options). "
        "Do NOT recalculate any numbers. Your job is to:\n"
        "1. Give a clear Go / Modify / Reschedule recommendation.\n"
        "2. Explain the main risk driver (heat, PM2.5, UV, or rain) in plain language.\n"
        "3. Present the suggested indoor alternatives and best time slots clearly.\n"
        "4. Be empathetic, concise, and encouraging. Use Markdown formatting."
    )

    try:
        from google import genai
        client = genai.Client(api_key=api_key)
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=json.dumps(summary_payload, default=str),
            config=genai.types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=0.4,
                max_output_tokens=700,
            ),
        )
        return response.text
    except Exception:
        return None

In [12]:
def _risk_badge(level):
    return {"High": "🔴 High", "Moderate": "🟡 Moderate", "Lower": "🟢 Lower"}.get(level, level)

def _indoor_tag(is_indoor):
    return "[Indoor]" if is_indoor else "[Outdoor]"


def _build_markdown_report(summary):
    """Build a rich Markdown report from the structured summary payload."""
    s = summary
    lines = []

    lines.append(f"## {_risk_badge(s['risk_level'])} Risk Assessment")
    lines.append("")
    lines.append(f"**Venue:** {s['venue_name']} ({s['venue_source']})  ")
    lines.append(f"**Coordinates:** {s['lat']:.5f}, {s['lon']:.5f} | {s['when']}  ")
    lines.append(f"**Activity:** {s['activity']} — VE {s['ve']:.1f} L/min for {s['duration']} min")
    lines.append("")

    lines.append("### ☁️ Conditions")
    lines.append("")
    lines.append("| Metric | Value |")
    lines.append("|--------|-------|")
    lines.append(f"| PM2.5 | {s['pm25']:.1f} µg/m³ |")
    lines.append(f"| Apparent heat | {s['heat']:.1f} °C |")
    lines.append(f"| UV index | {s['uv']:.1f} |")
    lines.append(f"| Rain probability | {s['rain']:.0f}% |")
    lines.append("")

    lines.append("### 🫁 Inhaled Dose Estimate")
    lines.append("")
    lines.append(f"- Inhaled air: **{s['inhaled_air_m3']:.2f} m³**")
    lines.append(f"- Estimated PM2.5 inhaled: **{s['dose_ug']:.1f} µg** ({s['dose_ug']/1000:.3f} mg)")
    lines.append(f"- Cigarette comparison: **≈ {s['cigarettes']:.2f}** cigarette-equivalents ({PM25_UG_PER_CIGARETTE:g} µg/cig assumption)")
    lines.append("")

    if s["reasons"]:
        lines.append("### ⚠️ Risk Drivers")
        lines.append("")
        for reason in s["reasons"]:
            lines.append(f"- {reason}")
        lines.append("")

    # Alternative times
    if s.get("alt_times"):
        lines.append("### 🕐 Better Time Slots (next 48 h, waking hours only)")
        lines.append("")
        lines.append("| When | PM2.5 | Heat | UV | Rain |")
        lines.append("|------|-------|------|----|------|")
        for slot in s["alt_times"]:
            lines.append(f"| {slot['time']} | {slot['pm25']:.1f} µg/m³ | {slot['heat']:.1f} °C | {slot['uv']:.1f} | {slot['rain']:.0f}% |")
        lines.append("")

    # Alternative venues
    if s.get("alt_venues"):
        lines.append("### 📍 Nearby Venue Alternatives")
        lines.append("")
        lines.append("| Venue | Type | Tag | Distance | Risk |")
        lines.append("|-------|------|-----|----------|------|")
        for v in s["alt_venues"]:
            lines.append(f"| {v['venue']} | {v['type']} | {v['tag']} | {v['distance_km']:.1f} km | {_risk_badge(v['risk'])} |")
        lines.append("")

    lines.append("---")
    lines.append("*Confirm opening hours, access, shelter, and activity suitability with the venue operator.*")
    return "\n".join(lines)


def run_assessment(_=None):
    with result_box:
        result_box.clear_output()
        try:
            venue = resolve_location(location_text.value, latitude.value, longitude.value)
            when = datetime.strptime(when_local.value, "%Y-%m-%d %H:%M").replace(tzinfo=SINGAPORE_TZ)
            forecast = forecast_at(venue["lat"], venue["lon"], when)
            forecast["when"] = when
            pm25 = pm25_override.value if pm25_override.value >= 0 else forecast["pm25"]
            heat = heat_override.value if heat_override.value >= 0 else forecast["heat"]
            uv = uv_override.value if uv_override.value >= 0 else forecast["uv"]
            rain = rain_override.value if rain_override.value >= 0 else forecast["rain"]
            if any(pd.isna(value) for value in [pm25, heat, uv, rain]):
                raise ValueError("A required forecast value is missing; enter an override for that field.")
            if ve_l_min.value <= 0 or duration_min.value <= 0:
                raise ValueError("VE and duration must be greater than zero.")
            inhaled_air_m3 = ve_l_min.value * duration_min.value / 1000.0
            dose_ug = pm25 * inhaled_air_m3
            cigarettes = dose_ug / PM25_UG_PER_CIGARETTE
            level, reasons = assess_risk(pm25, heat, uv, rain)

            # ---------- Alternative times (waking hours, ±3 h preference) ----------
            preferred_hour = when.hour
            time_options = alternative_time(forecast, duration_min.value, preferred_hour=preferred_hour)
            alt_times_list = []
            for entry in time_options:
                # entry: (near_preferred, severity, ts, pm, heat_v, uv_v, rain_v)
                _, _, ts, p, h, u, r = entry
                alt_times_list.append({"time": ts.strftime("%a %d %b %H:%M"), "pm25": p, "heat": h, "uv": u, "rain": r})

            # ---------- Alternative venues (indoor-prioritised when risky) ----------
            prefer_indoor = (level in ("High", "Moderate")) or rain >= 50 or pm25 >= 12.1
            alt_venues_list = []
            try:
                candidates = alternative_venues(venue["lat"], venue["lon"], when, prefer_indoor=prefer_indoor)
                for item in candidates:
                    alt_venues_list.append({
                        "venue": item["venue"], "type": item["type"],
                        "tag": _indoor_tag(item["is_indoor"]),
                        "distance_km": item["distance_km"],
                        "risk": item["risk"],
                    })
            except Exception:
                pass  # facility dataset unavailable; alt_venues_list stays empty

            # ---------- Structured payload ----------
            summary_payload = {
                "venue_name": venue["name"], "venue_source": venue["source"],
                "lat": venue["lat"], "lon": venue["lon"],
                "when": when.strftime("%Y-%m-%d %H:%M %Z"),
                "activity": activity.value, "ve": ve_l_min.value, "duration": duration_min.value,
                "pm25": pm25, "heat": heat, "uv": uv, "rain": rain,
                "inhaled_air_m3": inhaled_air_m3, "dose_ug": dose_ug,
                "cigarettes": cigarettes,
                "risk_level": level, "reasons": reasons,
                "alt_times": alt_times_list,
                "alt_venues": alt_venues_list,
            }

            # ---------- Render ----------
            llm_text = generate_llm_summary(summary_payload)
            if llm_text:
                display(Markdown("## 🤖 Coach Summary\n\n" + llm_text))
                display(Markdown("---\n*Deterministic details below:*"))

            display(Markdown(_build_markdown_report(summary_payload)))

        except Exception as error:
            display(Markdown(f"**❌ Unable to assess this plan:** {error}"))

run_button.on_click(run_assessment)

## Data and interpretation notes

- Open-Meteo provides forecast PM2.5, apparent temperature, UV and precipitation probability. The forecast is hourly and the requested time is matched to the nearest hour.
- OneMap search is used only to resolve the entered venue name to a location. Set `ONEMAP_TOKEN` in the notebook environment to enable it; otherwise the latitude and longitude fields are used.
- Alternative venue candidates come from SportSG's public facility GeoJSON and NParks' Park Facilities GeoJSON via data.gov.sg. The nearest candidates are shown as options; their local weather is not assumed to be better.
- Dose = PM2.5 concentration × VE × duration, with VE converted from L/min to m³/min. The calculation assumes the entered VE is representative throughout the whole activity and estimates inhaled mass.
- Risk categories and cigarette equivalents are rough planning aids. They do not replace NEA advisories, heat-stress guidance, lightning warnings or personal medical advice.
- **GeoJSON caching:** SportSG and NParks datasets are cached to `./cache_sportsg.json` and `./cache_nparks.json` after the first download. Delete these files to force a refresh.
- **Indoor classification** uses keyword matching on venue names and details (gym, sports hall, squash, etc.). A facility labelled `[Indoor]` has not been individually verified.
- **LLM summary** is generated via the `google.genai` client when `LLM_API_KEY` is set. The model does not recalculate any metrics.